# 11 · Post-training candidate store (data only)

The candidates for the budget-tolerance LoRA: one image and one question-answer turn per row, from FineVision and PixMo,
filtered as in `docs/notes/post_training_data.md` (all filters except the teacher pass). GQA and TextVQA stay
evaluation only, so every candidate image is hashed against their eval images.

A separate notebook from `09_teacher_cache` because 09 selects GPUs and loads the model; this one needs neither.

**Before running:** build the store in a terminal (this notebook only reads it):

```
python scripts/build_post_data.py --config-name vqav2 --rows 50   # smoke
python scripts/build_post_data.py                                 # everything, resumable
```

No GPU is used; a warning that `CUDA_VISIBLE_DEVICES` is unset is expected.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import io
import os
import socket

os.environ["CUDA_VISIBLE_DEVICES"] = ""  # CPU only; before adaloop.api is imported

In [ ]:
from adaloop import api

ROOT, _ = api.setup_notebook("configs/mvp.yaml")
cfg = api.load_config(ROOT / "configs/post_training_data.yaml")

In [ ]:
print("host", socket.gethostname(), "| on Inara or PACE:", api.is_gpu_host())

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

In [ ]:
# Settings
STORE_DIR = None  # None: out_dir of the config; or the --out path of the build
STORE = api.post_data_dir(cfg, ROOT, STORE_DIR)
FACTOR = cfg["candidate_factor"]
VAL = cfg["val_per_bucket"]
N_SHOW = 2
N_REHASH = 200
STORE

### Step-1:- What the build did

One row per source: rows left after each filter, eval-overlap and duplicate drops, bytes read, time and files used.
`note` says why a source ended below its target.

In [ ]:
summary = pd.read_csv(STORE / "summary.csv")
summary[["bucket", "source_config", "target", "kept", "rows_read", "after_ratings", "after_answer", "after_image", "eval_dup_dropped", "dup_dropped", "files_used", "files_total", "seconds", "note"]]

In [ ]:
summary.bytes_read.sum() / 1e9, summary.seconds.sum() / 3600  # GB, hours

### Step-2:- The store

All columns except the image bytes.

In [ ]:
df = api.load_post_data(STORE).to_pandas()
df.head()

In [ ]:
df.pivot_table(index="bucket", columns="split", values="id", aggfunc="count", margins=True)

In [ ]:
df.groupby(["bucket", "answer_type"]).size()

### Step-3:- Samples

Two per bucket: image, question, answer.

In [ ]:
picks = df.groupby("bucket").sample(N_SHOW, random_state=cfg["seed"])
rows = api.load_post_data(STORE, ["id", "image"], ids=picks.id.tolist()).to_pylist()
images = {r["id"]: Image.open(io.BytesIO(r["image"])) for r in rows}
for bucket, group in picks.groupby("bucket"):
    fig, axes = plt.subplots(1, N_SHOW, figsize=(6 * N_SHOW, 4))
    for ax, row in zip(axes, group.itertuples()):
        ax.imshow(images[row.id])
        ax.set_title(f"{bucket} | {row.source_config} | {row.width}x{row.height}", fontsize=9)
        ax.axis("off")
    plt.show()
    for row in group.itertuples():
        print(f"{row.source_config} [{row.split}] Q: {row.question[:300]!r}\n    A: {row.answer[:300]!r}")

### Step-4:- Counts

Each bucket needs at least its final mix size plus its val items, and at most `candidate_factor` times the mix size.

In [ ]:
per_bucket = df.bucket.value_counts()
for bucket, spec in cfg["buckets"].items():
    n = per_bucket.get(bucket, 0)
    assert spec["target"] + VAL[bucket] <= n <= FACTOR * spec["target"], (bucket, n, spec["target"])
    for source in spec["sources"]:
        if "cap" in source:
            assert (df.source_config == source["config"]).sum() <= FACTOR * source["cap"], source
assert df[df.split == "val"].bucket.value_counts().to_dict() == VAL
assert df.id.is_unique
print(f"✓ counts: {len(df)} candidates, {(df.split == 'val').sum()} val, caps kept")

### Step-5:- No eval overlap

Every stored hash against every GQA / TextVQA image of dev, test and ctrl_train (images only). A sample of images is
hashed again to check the stored values.

In [ ]:
blocked = api.post_data_eval_hashes(cfg, ROOT, STORE)
overlap = [r.id for r in df.itertuples() if api.near_eval(int(r.phash, 16), int(r.dhash, 16), blocked, cfg["dedupe"])]
assert not overlap, overlap[:5]
print(f"✓ no eval overlap: {len(df)} candidates vs {len(blocked)} eval images")

In [ ]:
check = df.sample(min(N_REHASH, len(df)), random_state=cfg["seed"])
stored = api.load_post_data(STORE, ["id", "image", "phash"], ids=check.id.tolist()).to_pylist()
same = [(api.phash(Image.open(io.BytesIO(r["image"])).convert("RGB")) ^ int(r["phash"], 16)).bit_count() for r in stored]
assert len(same) == len(check) and max(same) <= 4, max(same)  # a PNG re-encoded to JPEG may move a bit or two
print(f"✓ stored hashes match the stored images on {len(same)} samples (largest distance {max(same)})")

### Step-6:- Train and val share no image

In [ ]:
key = df.phash + df.dhash
assert key.is_unique
assert not set(key[df.split == "train"]) & set(key[df.split == "val"])
print(f"✓ train {(df.split == 'train').sum()} / val {(df.split == 'val').sum()}: image-disjoint by hash")